# Projective 3D (G(4,0)) — Homogeneous Coordinates

**Part II · Geometric Algebra & Core** — Tutorial 05

This tutorial is a deep dive into `BasisP3`, the projective 3D algebra $G(4, 0)$. It
extends Euclidean 3D with a fourth basis vector $e_4$ that acts as the *homogeneous
coordinate*: a finite point is $x\,e_1 + y\,e_2 + z\,e_3 + e_4$, and the origin is
simply $e_4$.

By the end you will be able to:

- Understand the **homogeneous embedding** — why $e_4$ carries the origin.
- Build **points, directions, lines, and planes** as blades (via the geometry submodule).
- Form lines and planes by **joining points** with the outer product.
- **Project** 3D points through a camera origin onto an image plane.
- Apply **projective rotors** and see how P3 relates to Euclidean E3.
- Use the `Geometry` class and the `analyze()`/`create()` pipeline for round-trips.

> **Prerequisites:** [Tutorial 04](../04_euclidean_e3/) (vectors, bivectors, rotors).
> Geometric entities and operators are created through the `pytanga.geometry` submodule.


## 1. Setup

The imports mirror Tutorial 04, but with `BasisP3` and a couple of extra entity types.


In [1]:
import math

from pytanga import MV
from pytanga.basis import BasisP3
from pytanga.geometry import Direction, Geometry, Line, Plane, Point, Rotor

P3 = BasisP3()          # G(4, 0) — three Euclidean vectors plus the homogeneous e4
geo = Geometry(P3)      # binds the algebra; OPNS/IPNS read from P3.opns (default True)


## 2. The homogeneous embedding

`BasisP3` adds a fourth basis vector `e4` (the *homogeneous direction*) to the three
Euclidean basis vectors. `e4` squares to $+1$ and carries the origin: a **finite point**
is a Euclidean vector plus one unit of `e4`, while a **direction** (an ideal point at
infinity) has no `e4` component at all.

The named blades are `e1`–`e4`, the Euclidean trivector `e123`, and the pseudoscalar
`I = e1∧e2∧e3∧e4`. The full algebra has 16 blades; the rest are addressed by string.


In [2]:
e1, e2, e3, e4 = P3.e1, P3.e2, P3.e3, P3.e4
I4 = P3.I                       # pseudoscalar e1 ∧ e2 ∧ e3 ∧ e4

print("blades:", list(P3.blades().keys()))
print()

(e4 * e4).show("e4 * e4  (the homogeneous direction squares to +1)")
geo(Point(0, 0, 0)).show("origin   (the point at the origin is just e4)")

# A finite point is its Euclidean vector plus one unit of e4
P3("2 e1 + 3 e2 + e3 + e4").show("Point(2, 3, 1)")


blades: ['e1', 'e2', 'e3', 'e4', 'e123', 'I']



e4 * e4  (the homogeneous direction squares to +1): 1

origin   (the point at the origin is just e4): e4

Point(2, 3, 1): 2 e1 + 3 e2 + e3 + e4

## 3. Points and directions

The `Geometry` class builds these blades for you. A `Point` is a grade-1 vector whose
`e4` coefficient is $1$; a `Direction` is the same vector with `e4` coefficient $0$ — the
ideal (infinite) point in that direction.


In [3]:
# Finite point — e4 coefficient 1
p = geo(Point(1, 2, 3))
p.show("Point(1, 2, 3)")

# Ideal point (direction) — e4 coefficient 0
d = geo(Direction(4, 0, 0))
d.show("Direction(4, 0, 0)")

print("analyze →", geo.analyze(p))
print("analyze →", geo.analyze(d))


Point(1, 2, 3): e1 + 2 e2 + 3 e3 + e4

Direction(4, 0, 0): 4 e1

analyze → Point(1.00, 2.00, 3.00)
analyze → Dir(4.00, 0.00, 0.00)


## 4. Lines and planes — the outer product

In projective geometry the **outer product** of points spans the subspace through
them: two points span a **line** (a grade-2 bivector), three points span a **plane**
(a grade-3 trivector) — the projective counterpart of the Euclidean "line through
two points". The general `join()` / `meet()` operations, which also handle subspaces
that overlap, are covered in a later section.


In [4]:
A = P3("e1 + e4")       # point (1, 0, 0)
B = P3("e2 + e4")       # point (0, 1, 0)
C = P3("e3 + e4")       # point (0, 0, 1)

(A ^ B).show("A ∧ B       (line  = outer product of two points)")
(A ^ B ^ C).show("A ∧ B ∧ C   (plane = outer product of three points)")


A ∧ B       (line  = outer product of two points): e12 + e14 - e24

A ∧ B ∧ C   (plane = outer product of three points): e123 + e124 - e134 + e234

## 5. Lines and planes via the geometry submodule

The same blades are produced by the `Line` and `Plane` dataclasses — `geo.create(...)`
computes the outer-product representation for you.


In [5]:
line = geo.create(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)))
plane = geo.create(Plane(point=Point(0, 0, 5), normal=Direction(0, 0, 1)))

line.show("Line through origin along x (bivector)")
plane.show("Plane z = 5 (trivector)")

print("analyze →", geo.analyze(line))
print("analyze →", geo.analyze(plane))


Line through origin along x (bivector): - e14

Plane z = 5 (trivector): 5 e123 + e124

analyze → Line(org=Point(-0.00, -0.00, -0.00), dir=Dir(1.00, 0.00, 0.00))
analyze → Plane(pt=Point(0.00, 0.00, 5.00), n=Dir(0.00, 0.00, 1.00))


## 6. Join and meet

The outer product *joins* two subspaces when they are independent, but it vanishes when
one subspace lies inside the other. `pytanga` provides two dedicated blade operations for
the general case:

- **`join(a, b)`** — the **union**: the smallest-grade blade that contains both `a` and
  `b` (found via blade factorization), even when the outer product is zero.
- **`meet(a, b)`** — the **intersection**: the largest-grade blade contained in both,
  i.e. `dual(join(dual(a), dual(b)))`.

Both return a *normalized* blade. They contrast with the **outer product** (the join
*only* for independent subspaces — zero for incident ones) and the **inner product** (the
metric: orthogonality, not containment).


In [6]:
A = P3("e1 + e4")     # point (1, 0, 0)
B = P3("e2 + e4")     # point (0, 1, 0)
L = A ^ B             # the line through A and B

# A lies on L, so the outer product vanishes …
(A ^ L).show("A ∧ L   (A lies on L → zero)")

# … but join() recovers the union — the line itself (normalized)
P3.join(A, L).show("join(A, L)")


A ∧ L   (A lies on L → zero): 0

join(A, L): 0.5774 e12 + 0.5774 e14 - 0.5774 e24

In [7]:
C = P3("e3 + e4")          # point (0, 0, 1)
plane1 = A ^ B ^ C         # the plane through A, B, C
plane2 = P3.e4 ^ A ^ B     # another plane — through the origin and the line L

# meet of a line and a plane that contains it → the line
P3.meet(L, plane1).show("meet(L, plane1)")

# meet of two planes → their common line (through A and B)
P3.meet(plane1, plane2).show("meet(plane1, plane2)")


meet(L, plane1): 0.5774 e12 + 0.5774 e14 - 0.5774 e24

meet(plane1, plane2): - 0.5774 e12 - 0.5774 e14 + 0.5774 e24

### Incidence relations

The outer product doubles as an **incidence test**: `A ∧ B = 0` exactly when one subspace
lies inside the other (e.g. a point on a line). The inner product, in contrast, is the
**metric** — it tests orthogonality, not containment.


In [8]:
# Outer product = 0  ⟺  incidence (containment)
print("A ∧ L      == 0 :", (A ^ L).is_zero, "   → A lies on L")
print("C ∧ L      == 0 :", (C ^ L).is_zero, "   → C does not lie on L")
print("C ∧ plane1 == 0 :", (C ^ plane1).is_zero, "   → C lies on plane1")

# Inner product = 0  ⟺  orthogonality (metric), not containment
print("e1 | e2    == 0 :", (e1 | e2).is_zero, "   → e1 ⊥ e2")
print("e1 | e1    == 0 :", (e1 | e1).is_zero, "   → e1 is not orthogonal to itself")


A ∧ L      == 0 : True    → A lies on L
C ∧ L      == 0 : False    → C does not lie on L
C ∧ plane1 == 0 : True    → C lies on plane1
e1 | e2    == 0 : True    → e1 ⊥ e2
e1 | e1    == 0 : False    → e1 is not orthogonal to itself


## 7. Perspective projection

A pinhole camera is a point (the optical center) plus an image plane. Projecting a world
point $P$ onto the plane means:

1. build the **ray** — the line through the camera and $P$: `ray = origin ∧ P`;
2. **meet** the ray with the image plane — the intersection point.

The meet (regressive product) is computed through duality:
`meet(A, B) = (A.dual() ∧ B.dual()).dual()` — wedge the duals, then dualize back.


In [9]:
# A camera at the origin with an image plane at z = 1
origin = geo(Point(0, 0, 0))
plane = geo(Point(0, 0, 1)) ^ geo(Point(1, 0, 1)) ^ geo(Point(0, 1, 1))
plane.show("image plane z = 1")

# A world point, and the ray from the camera through it
P = geo(Point(2, 4, 2))
ray = origin ^ P
ray.show("ray = origin ∧ P")

# Meet the ray with the image plane (regressive product via duals)
proj = (ray.dual() ^ plane.dual()).dual()
proj.show("meet(ray, plane)")
print("analyze →", geo.analyze(proj), "   (homogeneous: divide by the e4 weight)")


image plane z = 1: e123 + e124

ray = origin ∧ P: - 2 e14 - 4 e24 - 2 e34

meet(ray, plane): - 2 e1 - 4 e2 - 2 e3 - 2 e4

analyze → Point(1.00, 2.00, 1.00)    (homogeneous: divide by the e4 weight)


## 8. Projective rotors and the Euclidean relationship

A `Rotor` in P3 has exactly the same blade form as in E3 — it acts on the Euclidean part
and leaves the `e4` weight untouched. That is the concrete link between the two algebras:
drop `e4` from a P3 vector and you recover the E3 vector, and every E3 rotor is a P3
rotor.


In [10]:
# Identical form to an E3 rotor: scalar + bivector, acting on the Euclidean part
R = geo(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
R.show("Rotor(90° about z)")

p = geo(Point(1, 0, 0))
(R * p * ~R).show("R p ~R   (the e4 weight stays 1)")
print("analyze →", geo.analyze(R * p * ~R))

# The e1/e2/e3 coefficients are the Euclidean coordinates; e4 is just the weight
q = P3("3 e1 + 4 e2 + 5 e3 + e4")
print()
print("Euclidean coords:", q["e1"], q["e2"], q["e3"], "| e4 weight:", q["e4"])


Rotor(90° about z): 0.7071 - 0.7071 e12

R p ~R   (the e4 weight stays 1): e2 + e4

analyze → Point(0.00, 1.00, 0.00)

Euclidean coords: 3.0 4.0 5.0 | e4 weight: 1.0


## 9. A note on translation

Pure translation needs a **null** vector, which P3 does not have ($e_4^2 = +1$). The
`Translator` operator therefore raises in P3 — translations are handled by the conformal
model (`BasisN3`, [Tutorial 06](../06_conformal_n3/)) or by PGA3
([Tutorial 07](../07_pga3/)), not by the homogeneous model.


In [11]:
from pytanga.geometry import Translator

try:
    geo(Translator(vector=Direction(1, 0, 0)))
except ValueError as exc:
    print("ValueError:", exc)


ValueError: Translators require conformal embedding (N3); not available in P3.


## 10. The `analyze()` / `create()` pipeline

The geometry submodule is **bidirectional**: `create()` builds a multivector from a
geometric description, `analyze()` extracts the geometry back out. Every P3 entity
round-trips cleanly.


In [12]:
for name, e in [
    ("Point", Point(3, -1, 2)),
    ("Direction", Direction(0, 1, 0)),
    ("Line", Line(Point(0, 0, 0), Direction(1, 1, 0))),
    ("Plane", Plane(Point(1, 0, 0), Direction(0, 1, 0))),
]:
    result = geo.analyze(geo.create(e))
    print(f"{name:9s} → {result}")


Point     → Point(3.00, -1.00, 2.00)
Direction → Dir(0.00, 1.00, 0.00)
Line      → Line(org=Point(0.00, 0.00, 0.00), dir=Dir(0.71, 0.71, 0.00))
Plane     → Plane(pt=Point(-0.00, -0.00, -0.00), n=Dir(0.00, 1.00, 0.00))


## 11. Visual examples

`pytanga.viz` analyses P3 multivectors on the way into the viewer: a grade-1 point
renders as a **point**, a grade-2 bivector as a **line**, and a grade-3 trivector as a
**plane**. Viewer setup is covered in [Part I — Visualization](../../visualization/).


In [13]:
from pytanga.viz import Visualizer

# A point, a line (bivector), and a plane (trivector) in projective space
pnt = geo(Point(1, 2, 3))
lin = geo.create(Line(origin=Point(-2, -2, 0), direction=Direction(1, 1, 0.5)))
pln = geo.create(Plane(point=Point(0, 0, 2), normal=Direction(0, 0, 1)))

viz = Visualizer(title="P3 — point, line, and plane in projective space")
viz.new(pnt, color="#ff4444", label="P3 point")
viz.new(lin, color="#44ff44", label="P3 line (bivector)")
viz.new(pln, color="#4488ff", opacity=0.3, label="P3 plane (trivector)")

viz.display_snapshot()   # renders the scene inline (serverless)


In [14]:
# Perspective projection: a camera at the origin, an image plane at z = 1
origin = geo(Point(0, 0, 0))
plane = geo(Point(0, 0, 1)) ^ geo(Point(1, 0, 1)) ^ geo(Point(0, 1, 1))

world = geo(Point(2, 4, 2))
ray = origin ^ world
proj = (ray.dual() ^ plane.dual()).dual()

viz = Visualizer(title="P3 — perspective projection onto an image plane")
viz.new(world, color="#ff4444", label="world point P")
viz.new(origin, color="#ffffff", label="camera (origin)")
viz.new(ray, color="#44ff44", label="ray = origin ∧ P")
viz.new(plane, color="#4488ff", opacity=0.2, label="image plane z = 1")
viz.new(proj, color="#ffcc00", label="projected point")

viz.display_snapshot()


In [15]:
import os

# Standalone export: self-contained HTML, openable in any browser
os.makedirs("_output/05_projective_p3", exist_ok=True)
viz.export_snapshot("_output/05_projective_p3/projection.html", overwrite=True)
print("Exported _output/05_projective_p3/projection.html")


Exported _output/05_projective_p3/projection.html

## 12. Summary & next steps

You now know the homogeneous projective model `BasisP3`:

| Concept | API |
|---|---|
| Homogeneous point / direction | `geo(Point(...))` / `geo(Direction(...))` |
| Origin / ideal point | `e4` / a vector with no `e4` component |
| Line (join of 2 points) | `A ^ B` / `geo.create(Line(...))` |
| Plane (join of 3 points) | `A ^ B ^ C` / `geo.create(Plane(...))` |
| Ray / perspective projection | `origin ^ P` then `(ray.dual() ^ plane.dual()).dual()` |
| Projective rotor | `geo(Rotor(angle, Direction(...)))` |
| Round-trip | `geo.analyze(geo.create(entity))` |

**Where to go next:**

- [**06 · Conformal 3D**](../06_conformal_n3/) — the null-vector embedding that adds
  spheres, circles, and translations.
- [**07 · PGA 3D**](../07_pga3/) — plane-based geometric algebra with translators/motors.
- [**08 · Duality & Complements**](../08_duality/) — the `dual()`/`ldual()`/`complement()`
  family used by the meet above.
